# Prepare Annotations for InferCNV

##### Franziska Niemeyer, 2026-03-27

In [ ]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import anndata as ad
import numpy as np
import os
import math
import sys
import cmcrameri.cm as cmc

from pathlib import Path
from aquarel import load_theme
from scipy.sparse import issparse

In [ ]:
WORKING_DIR = ".."
OUT_DIR = f"{WORKING_DIR}/cnv_inference"
ADATA = "../../../quality_control/primary-cohort/adata.h5ad"

adata = ad.read_h5ad(ADATA)

In [ ]:
adata = adata[~adata.obs['patient'].isin(['Marker'])].copy()

In [ ]:
adata.obs['patient'].value_counts()

In [ ]:
sample_key = "sample"
annotation_key = "histology"

# keep only spots with a non-missing annotation
adata_filt = adata[
    (adata.obs[annotation_key].notna())
].copy()

print(adata_filt)

In [ ]:
adata.obs['histology'].value_counts()

In [ ]:
# keep the current gene symbols
adata.var["gene_symbol"] = adata_filt.var_names.astype(str)

# switch var_names to Ensembl IDs
adata.var_names = adata_filt.var["gene_ids"].astype(str)

print(adata.var_names.is_unique)
print(adata.var_names[:5])

In [ ]:
out_dir = os.path.join(OUT_DIR, "infercnv_input")
os.makedirs(out_dir, exist_ok=True)

filtered_h5ad_path = os.path.join(out_dir, "visium_filtered_for_infercnv.h5ad")
adata.write(filtered_h5ad_path)

print(f"Saved filtered AnnData to: {filtered_h5ad_path}")

In [ ]:
def export_infercnv_annotations_per_sample(
    adata,
    out_dir,
    sample_key="slide",
    annotation_key="annotation",
    use_full_obs_names=False,
    obs_name_sep="_",
):
    """
    Export one inferCNV annotation file per sample.
    """
    out_dir = out_dir
    os.makedirs(out_dir, exist_ok=True)

    for sample in adata.obs[sample_key].unique():
        ad = adata[adata.obs[sample_key] == sample].copy()

        if use_full_obs_names:
            barcodes = ad.obs_names.to_series(index=ad.obs_names)
        else:
            def strip_prefix(x):
                prefix = f"{sample}{obs_name_sep}"
                return x[len(prefix):] if x.startswith(prefix) else x

            barcodes = pd.Series(
                [strip_prefix(x) for x in ad.obs_names],
                index=ad.obs_names
            )

        ann_df = pd.DataFrame({
            "barcode": barcodes.values,
            "annotation": ad.obs[annotation_key].astype(str).values
        })

        out_path = os.path.join(out_dir, f"{sample}_infercnv_annotations.tsv")
        print(ann_df.head(3))
        ann_df.to_csv(out_path, sep="\t", header=False, index=False)

        print(f"{sample}: wrote {ann_df.shape[0]} annotations -> {out_path}")

In [ ]:
ann_dir = os.path.join(out_dir, "annotations")

export_infercnv_annotations_per_sample(
    adata=adata,
    out_dir=ann_dir,
    sample_key="sample",
    annotation_key="histology",
    use_full_obs_names=True,
)

In [ ]:
from pathlib import Path
import gzip
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.io import mmwrite


def export_infercnv_inputs_per_sample(
    adata,
    out_dir,
    sample_key="slide",
    annotation_key="annotation",
    layer=None,
    use_full_obs_names=False,
    obs_name_sep="_",
    write_mtx=True,
    write_tsv=False,
    write_h5=True,
):
    """
    Export per-sample inferCNV input files from an AnnData object.

    For each sample writes:
      - <sample>_infercnv_annotations.tsv
      - <sample>_counts.mtx.gz
      - <sample>_barcodes.tsv.gz
      - <sample>_genes.tsv.gz
      - optionally <sample>_counts.tsv.gz
      - optionally <sample>_counts.h5   (custom HDF5, not native inferCNV input)

    Notes
    -----
    - inferCNV expects raw counts, not log-normalized values.
    - The annotation file row order matches the count matrix columns.
    """

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for sample in adata.obs[sample_key].unique():
        ad = adata[adata.obs[sample_key] == sample].copy()

        # choose matrix
        if layer is not None:
            X = ad.layers[layer]
        else:
            X = ad.X

        # make sure counts are sparse CSR for efficient column export
        if sp.issparse(X):
            X = X.tocsr()
        else:
            X = sp.csr_matrix(np.asarray(X))

        # inferCNV usually expects genes x cells/spots
        X_gc = X.T.tocsr()   # genes x spots

        # barcodes: either full obs_names or strip sample prefix
        if use_full_obs_names:
            barcodes = np.array(ad.obs_names.astype(str))
        else:
            prefix = f"{sample}{obs_name_sep}"
            barcodes = np.array([
                x[len(prefix):] if str(x).startswith(prefix) else str(x)
                for x in ad.obs_names
            ])

        annotations = np.array(ad.obs[annotation_key].astype(str))
        genes = np.array(ad.var_names.astype(str))
        print(f"Genes look like: {genes[:3]}")

        # ---------- annotation file ----------
        ann_df = pd.DataFrame({
            "barcode": barcodes,
            "annotation": annotations,
        })

        ann_path = out_dir / f"{sample}_infercnv_annotations.tsv"
        ann_df.to_csv(ann_path, sep="\t", header=False, index=False)

        # ---------- barcodes / genes ----------
        barcode_path = out_dir / f"{sample}_barcodes.tsv.gz"
        gene_path = out_dir / f"{sample}_genes.tsv.gz"

        with gzip.open(barcode_path, "wt") as f:
            for b in barcodes:
                f.write(f"{b}\n")

        with gzip.open(gene_path, "wt") as f:
            for g in genes:
                f.write(f"{g}\n")

        # ---------- Matrix Market ----------
        if write_mtx:
            mtx_path = out_dir / f"{sample}_counts.mtx"
            mmwrite(str(mtx_path), X_gc)

            # gzip it
            with open(mtx_path, "rb") as f_in, gzip.open(f"{mtx_path}.gz", "wb") as f_out:
                f_out.writelines(f_in)
            mtx_path.unlink()

        # ---------- dense TSV ----------
        if write_tsv:
            # This can get very large; use only if you really want inferCNV-style text matrix
            tsv_path = out_dir / f"{sample}_counts.tsv.gz"

            if sp.issparse(X_gc):
                dense = X_gc.toarray()
            else:
                dense = np.asarray(X_gc)

            df_counts = pd.DataFrame(dense, index=genes, columns=barcodes)
            df_counts.to_csv(tsv_path, sep="\t", compression="gzip")

        # ---------- custom HDF5 ----------
        if write_h5:
            h5_path = out_dir / f"{sample}_counts.h5"
            with h5py.File(h5_path, "w") as h5:
                h5.create_dataset("counts", data=X_gc.toarray(), compression="gzip")
                h5.create_dataset("genes", data=genes.astype("S"))
                h5.create_dataset("barcodes", data=barcodes.astype("S"))
                h5.create_dataset("annotations", data=annotations.astype("S"))

        print(
            f"{sample}: wrote {ad.n_obs} spots, {ad.n_vars} genes "
            f"-> annotations + counts files"
        )

In [ ]:
export_infercnv_inputs_per_sample(
    adata=adata,
    out_dir=os.path.join(out_dir, "per_sample"),
    sample_key="sample",
    annotation_key="histology",
    layer="counts",
    use_full_obs_names=True,
    write_mtx=False,
    write_tsv=False,
    write_h5=True,
)

In [ ]:
from pathlib import Path
import gzip
import numpy as np
import pandas as pd
import scipy.sparse as sp
import h5py
from scipy.io import mmwrite


def export_infercnv_inputs_combined(
    adata,
    out_dir,
    sample_key="slide",
    annotation_key="annotation",
    patient_key="sample",
    reference_annotation="Stroma",
    layer=None,
    prefix="combined",
    use_full_obs_names=False,
    obs_name_sep="_",
    write_mtx=True,
    write_tsv=False,
    write_h5=True,
):
    """
    Export combined inferCNV input files from an AnnData object for multiple samples together.

    For all samples combined, it writes:
      - infercnv_annotations.tsv
      - counts.mtx.gz
      - barcodes.tsv.gz
      - genes.tsv.gz
      - optionally counts.tsv.gz
      - optionally counts.h5   (custom HDF5, not native inferCNV input)

    Annotation handling
    -------------------
    All non-reference annotations are exported as:
        {patient_id}_{annotation}

    The reference annotation is kept pooled across patients as:
        {reference_annotation}

    Example
    -------
    If patient_key="sample" and reference_annotation="Stroma":
        Tumor epithelium -> P001_Tumor epithelium
        Benign epithelium -> P001_Benign epithelium
        Stroma -> Stroma

    Notes
    -----
    - inferCNV expects raw counts, not log-normalized values.
    - The annotation file row order matches the count matrix columns.
    """

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # choose matrix
    if layer is not None:
        X = adata.layers[layer]
    else:
        X = adata.X

    # make sure counts are sparse CSR
    if sp.issparse(X):
        X = X.tocsr()
    else:
        X = sp.csr_matrix(np.asarray(X))

    # inferCNV expects genes x cells/spots
    X_gc = X.T.tocsr()  # genes x spots

    # barcodes
    if use_full_obs_names:
        barcodes = np.array(adata.obs_names.astype(str))
    else:
        sample_names = adata.obs[sample_key].astype(str).values
        stripped_barcodes = []

        for obs_name, sample in zip(adata.obs_names.astype(str), sample_names):
            obs_prefix = f"{sample}{obs_name_sep}"
            if obs_name.startswith(obs_prefix):
                stripped_barcodes.append(obs_name[len(obs_prefix):])
            else:
                stripped_barcodes.append(obs_name)

        stripped_barcodes = np.array(stripped_barcodes)

        if pd.Index(stripped_barcodes).duplicated().any():
            print(
                "[warn] Stripped barcodes are not unique across samples. "
                "Using full obs_names instead."
            )
            barcodes = np.array(adata.obs_names.astype(str))
        else:
            barcodes = stripped_barcodes

    # genes
    genes = np.array(adata.var_names.astype(str))

    # build inferCNV annotations:
    # - pooled reference group
    # - patient-specific labels for everything else
    raw_annotations = adata.obs[annotation_key].astype(str)
    patient_ids = adata.obs[patient_key].astype(str)

    export_annotations = np.where(
        raw_annotations == reference_annotation,
        reference_annotation,
        patient_ids + " - " + raw_annotations
    )

    print(f"Combined object: {adata.n_obs} spots, {adata.n_vars} genes")
    print(f"Genes look like: {genes[:3]}")
    print("Example exported annotation labels:")
    print(pd.Series(export_annotations).value_counts().head(10))

    # ---------- annotation file ----------
    ann_df = pd.DataFrame({
        "barcode": barcodes,
        "annotation": export_annotations,
    })

    ann_path = out_dir / f"{prefix}_infercnv_annotations.tsv"
    ann_df.to_csv(ann_path, sep="\t", header=False, index=False)

    # ---------- barcodes / genes ----------
    barcode_path = out_dir / f"{prefix}_barcodes.tsv.gz"
    gene_path = out_dir / f"{prefix}_genes.tsv.gz"

    with gzip.open(barcode_path, "wt") as f:
        for b in barcodes:
            f.write(f"{b}\n")

    with gzip.open(gene_path, "wt") as f:
        for g in genes:
            f.write(f"{g}\n")

    # ---------- Matrix Market ----------
    if write_mtx:
        mtx_path = out_dir / f"{prefix}_counts.mtx"
        mmwrite(str(mtx_path), X_gc)

        with open(mtx_path, "rb") as f_in, gzip.open(f"{mtx_path}.gz", "wb") as f_out:
            f_out.writelines(f_in)
        mtx_path.unlink()

    # ---------- dense TSV ----------
    if write_tsv:
        tsv_path = out_dir / f"{prefix}_counts.tsv.gz"

        if sp.issparse(X_gc):
            dense = X_gc.toarray()
        else:
            dense = np.asarray(X_gc)

        df_counts = pd.DataFrame(dense, index=genes, columns=barcodes)
        df_counts.to_csv(tsv_path, sep="\t", compression="gzip")

    # ---------- custom HDF5 ----------
    if write_h5:
        h5_path = out_dir / f"{prefix}_counts.h5"
        with h5py.File(h5_path, "w") as h5:
            if sp.issparse(X_gc):
                counts = X_gc.toarray()
            else:
                counts = np.asarray(X_gc)

            h5.create_dataset("counts", data=counts, compression="gzip")
            h5.create_dataset("genes", data=genes.astype("S"))
            h5.create_dataset("barcodes", data=barcodes.astype("S"))
            h5.create_dataset("annotations", data=np.asarray(export_annotations).astype("S"))

    print(
        f"Wrote combined inferCNV input files for {adata.n_obs} spots and "
        f"{adata.n_vars} genes to: {out_dir}"
    )

In [ ]:
export_infercnv_inputs_combined(
    adata=adata,
    out_dir=os.path.join(out_dir, "combined"),
    sample_key="sample",
    annotation_key="histology",
    layer="counts",
    prefix="combined_stroma_ref",
    patient_key="patient",
    reference_annotation="Other",
    use_full_obs_names=True,
    write_mtx=True,
    write_tsv=False,
    write_h5=True,
)